# Analyse séquentielle sur des données de métro

Vincent Guigue (vincent.guigue@agroparistech.fr)

Le jeu de données du métro de Hangzhou décrit le flux entrant et sortant de 80 stations de la ville, agrégé par quart d'heure entre $5h30$ et $23h30$ chaque jour. Deux tenseurs sont dans l'archive, un d'apprentissage et l'autre de test. Ils sont de taille $D\times T \times S \times 2$ avec $D$ le nombre de jours, $T=73$ les tranches successives de quart d'heure entre $5h30$ et $23h30$, $S=80$ le nombre de stations et les flux entrant et sortant pour la dernière dimension.

Deux tâches sont traitées :
- **Partie B, classification (*many-to-one*)** : à partir d'une séquence de flux observée dans une station, prédire **de quelle station** il s'agit ;
- **Partie C, prévision (*many-to-many*)** : à partir des flux passés de toutes les stations, prédire les flux futurs.

On travaille sur un **sous-échantillon des stations pour simplifier les choses** : ce sous-échantillon est paramétré dans les boîtes ci-dessous.

Le réseau récurrent est fourni (classe `RNN` du fichier annexe `utils.py`, à lire !) ; l'enjeu est d'adapter la boucle d'apprentissage standard à cette architecture.

## Table des matières

- [0. Mise en place : librairies, device, TensorBoard](#0-mise-en-place-librairies-device-tensorboard)
- [A. Chargement d'un sous-échantillon des données](#a-chargement-d-un-sous-echantillon-des-donnees)
- [B. Classification de stations (*many-to-one*)](#b-classification-de-stations-many-to-one)
    - [B.1 Définition du modèle](#b-1-definition-du-modele)
    - [B.2 Apprentissage](#b-2-apprentissage)
    - [B.3 Évaluation des performances](#b-3-evaluation-des-performances)
- [C. Prédiction d'affluence (*many-to-many*)](#c-prediction-d-affluence-many-to-many)
    - [C.1 Analyser en détail les dimensions des données](#c-1-analyser-en-detail-les-dimensions-des-donnees)
    - [C.2 Définition du modèle](#c-2-definition-du-modele)
    - [C.3 Prévision à plusieurs pas de temps](#c-3-prevision-a-plusieurs-pas-de-temps)
    - [C.4 Apprentissage](#c-4-apprentissage)
- [D. Métriques et exploitation des résultats](#d-metriques-et-exploitation-des-resultats)
    - [D.1 Quelle métrique ?](#d-1-quelle-metrique)
    - [D.2 Métriques en fonction de l'horizon de prédiction](#d-2-metriques-en-fonction-de-l-horizon-de-prediction)

<a id="0-mise-en-place-librairies-device-tensorboard"></a>
## 0. Mise en place : librairies, device, TensorBoard

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader

from utils import RNN, SampleMetroDataset, ForecastMetroDataset  # code fourni

In [ ]:
# ---- Choix du device ----
# Pour ces petits RNN, la récurrence est une boucle séquentielle sur le temps : sur Mac, le GPU (MPS)
# est en général PLUS LENT que le CPU (mesuré sur le TP métro : 12 s/epoch en MPS contre 1,2 s/epoch en CPU).
# MPS n'est donc activé que sur demande. Avec un GPU NVIDIA (CUDA), le gain reste réel.
USE_MPS = False

if torch.cuda.is_available():
    device = torch.device("cuda")
elif USE_MPS and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("torch :", torch.__version__, "| device :", device)
_ = torch.manual_seed(0)  # reproductibilité

In [ ]:
# ---- TensorBoard ----
from pathlib import Path
TB_PATH = Path("/tmp/logs/module4")    # une expérience = un sous-répertoire : SummaryWriter(TB_PATH / nom)
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:                    # tensorboard absent : on continue sans enregistrer les courbes
    class SummaryWriter:
        def __init__(self, *args, **kwargs): pass
        def add_scalar(self, *args, **kwargs): pass
        def close(self): pass
    print("tensorboard non installé : les courbes ne seront pas enregistrées")

# Option 1 (recommandée) : dans un terminal, lancer la commande ci-dessous puis ouvrir http://localhost:6006
print(f"tensorboard --logdir {TB_PATH}")

# Option 2 : TensorBoard directement dans le notebook (décommenter les 2 lignes)
# %load_ext tensorboard
# %tensorboard --logdir {TB_PATH}

<a id="a-chargement-d-un-sous-echantillon-des-donnees"></a>
## A. Chargement d'un sous-échantillon des données

Dans la partie B, un exemple est un couple (séquence de `LENGTH` quarts d'heure des flux d'**une** station, numéro de cette station). Lire le code de `SampleMetroDataset` dans `utils.py` pour comprendre comment sont construits les exemples et comment les données sont normalisées.

In [ ]:
PATH = "./data/"
CLASSES    = 10    # nombre de stations utilisées (= nombre de classes)
LENGTH     = 20    # longueur des séquences
DIM_INPUT  = 2     # dimension de l'entrée : 1 (flux entrant) ou 2 (entrant / sortant)
BATCH_SIZE = 32
LATENT     = 10    # taille de l'état caché

matrix_train, matrix_test = torch.load(PATH + "hzdataset.pch")
print("train :", matrix_train.size(), "| test :", matrix_test.size())

ds_train = SampleMetroDataset(matrix_train[:, :, :CLASSES, :DIM_INPUT], length=LENGTH)
# ATTENTION : le test est normalisé avec les statistiques de l'apprentissage
ds_test  = SampleMetroDataset(matrix_test[:, :, :CLASSES, :DIM_INPUT], length=LENGTH, stations_max=ds_train.stations_max)
data_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
data_test  = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False)

Analyser les données : nombre d'exemples, forme d'un exemple, répartition des classes...

In [ ]:
# quelques propositions... Mais à vous de jouer
print("nb exemples :", len(ds_train))
x, y = ds_train[0]
print("x :", x.size(), "| y :", y)

cl = [ds_train[i][1] for i in range(len(ds_train))]
print("classes :", np.unique(cl, return_counts=True))

<a id="b-classification-de-stations-many-to-one"></a>
## B. Classification de stations (*many-to-one*)

<a id="b-1-definition-du-modele"></a>
### B.1 Définition du modèle

**ATTENTION** : le modèle récurrent est défini dans le fichier joint `utils.py`. Notez que :
- le `forward` attend des séquences **temps en premier** (`length x batch x dim`), alors que le `DataLoader` fournit des batchs `batch x length x dim` ;
- il renvoie la séquence de **tous** les états cachés (`length x batch x latent`) ;
- le **décodeur** (couche de classification) est déclaré à part.

Cela a une incidence directe sur l'inférence du modèle, donc sur la boucle d'apprentissage.

In [ ]:
rnn = RNN(DIM_INPUT, LATENT)              # cf. code dans utils.py
decoder = nn.Linear(LATENT, CLASSES)
loss_fn = nn.CrossEntropyLoss()
# l'optimiseur doit voir les paramètres des DEUX modules
optim = torch.optim.Adam(list(rnn.parameters()) + list(decoder.parameters()), lr=1e-3)

<a id="b-2-apprentissage"></a>
### B.2 Apprentissage

L'enjeu est de faire rentrer ce réseau dans la boucle d'apprentissage classique développée dans les séances précédentes.

1. Jouer avec le modèle `rnn` pour comprendre ses entrées et ses sorties : vérifier les dimensions de sortie du réseau et **choisir où brancher le décodeur** (cellule ci-dessous) ;
2. Écrire la fonction `predict(rnn, decoder, x)` qui renvoie les logits pour un batch `x` issu du `DataLoader` ;
3. Compléter la boucle d'apprentissage (calquée sur celle des TP précédents) et lancer l'apprentissage.

Suivre les performances directement dans TensorBoard pendant l'apprentissage.

In [ ]:
# exploration des dimensions
x, y = next(iter(data_train))
print("x (DataLoader) :", x.size())
##  TODO 

In [ ]:
def predict(rnn, decoder, x):
    """ x : batch x length x dim  =>  logits : batch x classes """
    ##  TODO 


def evaluate(rnn, decoder, loader):
    """ renvoie (loss moyenne, accuracy) sur loader """
    cumloss, cumacc, count = 0., 0., 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            yhat = predict(rnn, decoder, x)
            cumloss += loss_fn(yhat, y).item() * len(y)
            cumacc  += (yhat.argmax(1) == y).sum().item()
            count   += len(y)
    return cumloss / count, cumacc / count

In [ ]:
def train(rnn, decoder, optim, epochs, train_loader, test_loader, name="predictStations"):
    writer = SummaryWriter(TB_PATH / f"{name}-{time.strftime('%Y%m%d-%H%M%S')}")
    rnn, decoder = rnn.to(device), decoder.to(device)
    for epoch in tqdm(range(epochs)):
        ##  TODO 
        # suivi des performances (on ré-évalue proprement l'apprentissage à la fin de chaque epoch)
        for split, loader in [("train", train_loader), ("test", test_loader)]:
            l, acc = evaluate(rnn, decoder, loader)
            writer.add_scalar(f"loss/{split}", l, epoch)
            writer.add_scalar(f"accuracy/{split}", acc, epoch)
    writer.close()

In [ ]:
# ~1 min sur CPU
train(rnn, decoder, optim, 30, data_train, data_test)

<a id="b-3-evaluation-des-performances"></a>
### B.3 Évaluation des performances

Comparer les performances en apprentissage et en test (taux de bonne classification) à l'issue de l'apprentissage. Quel est le taux de bonne classification d'un classifieur aléatoire ?

Le système présente-t-il des symptômes de sur-apprentissage ? Dans l'affirmative, quelle procédure aurait-on pu mettre en place ?

In [ ]:
print("train : loss %.3f | accuracy %.3f" % evaluate(rnn, decoder, data_train))
print("test  : loss %.3f | accuracy %.3f" % evaluate(rnn, decoder, data_test))
##  TODO 

<a id="c-prediction-d-affluence-many-to-many"></a>
## C. Prédiction d'affluence (*many-to-many*)

L'objectif de cette partie est de faire de la prédiction de séries temporelles : à partir d'une séquence de flux de longueur $t$ pour l'**ensemble des stations** du jeu de données, on veut prédire le flux à $t+1$, $t+2$, $\ldots$. Vous entraînerez un RNN commun à toutes les stations qui prend une série dans $\mathbb{R}^{S\times 2}$ et prédit une série dans $\mathbb{R}^{S\times 2}$.

Que doit-on changer au modèle précédent ? Quel coût est dans ce cas plus adapté que la cross-entropie ?

Dans ce contexte de réseau *many-to-many*, la supervision se fait à **chaque étape** de la séquence : à chaque instant $t$, on prédit $x_{t+1}$ à partir de $h_t$. La rétro-propagation n'est faite qu'une fois toute la séquence vue, mais à un instant $t$ le gradient prend en compte l'erreur à cet instant **et** l'erreur cumulée des pas de temps suivants.

**Apprentissage vs inférence** : en apprentissage, on donne au réseau la vraie valeur $x_t$ à chaque pas de temps (*teacher forcing*) ; pour prédire à plus long terme, on doit ré-injecter ses propres prédictions (fonction `forecast` fournie plus bas).

In [ ]:
STATIONS  = 20     # nombre de stations
LENGTH_C  = 20     # longueur des séquences
HORIZON   = 10     # horizon de prévision évalué
LATENT_C  = 20     # taille de l'état caché

ds_train_c = ForecastMetroDataset(matrix_train[:, :, :STATIONS, :DIM_INPUT], length=LENGTH_C)
# ATTENTION : normalisation du test avec les statistiques de l'apprentissage (stations_max est un tenseur S x 2)
ds_test_c  = ForecastMetroDataset(matrix_test[:, :, :STATIONS, :DIM_INPUT], length=LENGTH_C, stations_max=ds_train_c.stations_max)

data_train_c = DataLoader(ds_train_c, batch_size=BATCH_SIZE, shuffle=True)
data_test_c  = DataLoader(ds_test_c, batch_size=BATCH_SIZE, shuffle=False)

<a id="c-1-analyser-en-detail-les-dimensions-des-donnees"></a>
### C.1 Analyser en détail les dimensions des données

Extraire un exemple du dataset, mesurer les tailles de `x` et `y`, et tracer quelques séries. Votre code doit vous mener à quelque chose comme la figure ci-dessous :

![ts-metro](data/xy-teacherforcing.png)

In [ ]:
# Extraire un échantillon du dataset, mesurer les tailles de x et y... puis faire un plot
##  TODO 

<a id="c-2-definition-du-modele"></a>
### C.2 Définition du modèle

Le décodeur étant à part, prendre le temps de réfléchir à la dimension d'entrée du réseau récurrent (toutes les stations sont traitées ensemble) et à celle du décodeur. Choisir la loss adaptée.

In [ ]:
# rnn_c =
# decoder_c =
# loss_c =
##  TODO 
optim_c = torch.optim.Adam(list(rnn_c.parameters()) + list(decoder_c.parameters()), lr=1e-3)

<a id="c-3-prevision-a-plusieurs-pas-de-temps"></a>
### C.3 Prévision à plusieurs pas de temps

On fournit la fonction de prévision qui exploite la structure du réseau : après avoir lu un contexte, elle **ré-injecte la sortie prédite** pour prédire les valeurs suivantes.

In [ ]:
def forecast(rnn, decoder, x, length=10):
    """ x : T x batch x d (contexte)  =>  les `length` pas de temps suivants : length x batch x d """
    with torch.no_grad():
        h = rnn(x)[-1]                    # dernier état caché après lecture du contexte
        yhat = []
        for _ in range(length):
            x_next = decoder(h)           # prédiction du pas de temps suivant
            yhat.append(x_next)
            h = rnn.one_step(x_next, h)   # on ré-injecte la prédiction comme nouvelle entrée
    return torch.stack(yhat)


def to_seq(x):
    """ batch x T x S x dim  =>  T x batch x (S*dim) """
    return x.flatten(2).transpose(0, 1)

<a id="c-4-apprentissage"></a>
### C.4 Apprentissage

Compléter la boucle : en apprentissage, perte *teacher forcing* sur tous les pas de temps ; en test, on calcule en plus l'erreur de prévision à `HORIZON` pas de temps (avec `forecast`).

**Attention à l'alignement** : `x` contient les instants $t_0..t_0+T-1$ et `y` les instants $t_0+1..t_0+T$. La séquence complète est donc `torch.cat([x, y[-1:]])`.

In [ ]:
def train_forecast(rnn, decoder, optim, epochs, train_loader, test_loader, horizon=HORIZON, name="Forecast"):
    writer = SummaryWriter(TB_PATH / f"{name}-{time.strftime('%Y%m%d-%H%M%S')}")
    rnn, decoder = rnn.to(device), decoder.to(device)
    for epoch in tqdm(range(epochs)):
        ##  TODO 
    writer.close()

In [ ]:
# ~1 min sur CPU
train_forecast(rnn_c, decoder_c, optim_c, 100, data_train_c, data_test_c)

<a id="d-metriques-et-exploitation-des-resultats"></a>
## D. Métriques et exploitation des résultats

<a id="d-1-quelle-metrique"></a>
### D.1 Quelle métrique ?

Quelle métrique vous semble adaptée à ce problème ? La MSE sur des données normalisées est difficile à interpréter.

Ça vaut le coup de tester la MAPE (*mean absolute percentage error*) et de vérifier les problèmes autour des heures de faible affluence... pour ensuite regarder ce qui existe comme alternatives (SMAPE, WAPE...) :
- [SMAPE](https://en.wikipedia.org/wiki/Symmetric_mean_absolute_percentage_error)
- [explications sur les métriques de régression](https://kobia.fr/regression-metrics-quelle-metrique-choisir/)

<a id="d-2-metriques-en-fonction-de-l-horizon-de-prediction"></a>
### D.2 Métriques en fonction de l'horizon de prédiction

Utiliser la fonction `forecast` pour prédire à des horizons de 1 à `HORIZON`, calculer les métriques **sur les flux dé-normalisés** (en nombre de passagers) en fonction de l'horizon, et comparer à une **référence naïve** : la persistance (on prédit que le flux reste égal à la dernière valeur observée). Un modèle qui ne bat pas la persistance n'a aucun intérêt !

In [ ]:
##  TODO 